# Sillok-LM Remote Colab Runner (tsarkr Tunnel ↔ Colab Pro A100)

Google Colab에서 **런타임 -> 런타임 유형 변경 -> GPU (A100 권장)**를 선택한 뒤 아래 셀을 순서대로 실행하세요.

In [13]:
import os
import sys
import urllib.request
from pathlib import Path

# 0) GPU 확인
!nvidia-smi

# 1) 에지 호스트(tsarkr) Cloudflare 터널 주소
TUNNEL_URL = "https://reg-township-clan-founded.trycloudflare.com"

# 2) Google Drive 마운트 (체크포인트 영구 저장)
DRIVE_ROOT = Path("/content/drive/MyDrive/sillok_lm")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except ImportError:
    print("Not running inside Google Colab; skipping Drive mount.")

# 3) 코드 번들 수신 및 작업 디렉토리 설정
PROJECT_DIR = Path("/content/sillok")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
if not (PROJECT_DIR / "src" / "sillok").exists():
    urllib.request.urlretrieve(f"{TUNNEL_URL}/artifacts/sillok_code.tar.gz", "sillok_code.tar.gz")
    !tar -xzf sillok_code.tar.gz
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Working directory:", PROJECT_DIR)

Mon Nov 16 02:54:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             27W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# 4) 의존성 설치 및 런타임/GPU 진단
!pip install -q -e .

from sillok.remote import detect_runtime, format_runtime_report
print(format_runtime_report(detect_runtime()))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sillok-lm (pyproject.toml) ... done
Runtime Mode    : Google Colab Remote
Python          : 3.13.16
CPU             : Intel(R) Xeon(R) CPU @ 2.00GHz (2 cores)
System Memory   : 11.14 / 12.67 GiB available
PyTorch         : installed
CUDA / GPU      : Tesla T4 (14.56 GB, bf16=True)
Host Services   : sshd=True, vscode-server=False, cloudflared=inactive
Google Drive    : /content/drive/MyDrive


In [15]:
# 5) tsarkr 터널에서 전처리 완료된 train.bin (120MB), val.bin (2.4MB), tokenizer.json (2.6MB) 수신
from sillok.remote import download_from_tunnel

download_from_tunnel(TUNNEL_URL, "processed/train.bin", "data/processed/train.bin")
download_from_tunnel(TUNNEL_URL, "processed/val.bin", "data/processed/val.bin")
download_from_tunnel(TUNNEL_URL, "processed/dataset_meta.json", "data/processed/dataset_meta.json")
download_from_tunnel(TUNNEL_URL, "artifacts/tokenizer/tokenizer.json", "artifacts/tokenizer/tokenizer.json")
download_from_tunnel(TUNNEL_URL, "artifacts/tokenizer/tokenizer_meta.json", "artifacts/tokenizer/tokenizer_meta.json")
!ls -lh data/processed/ artifacts/tokenizer/

artifacts/tokenizer/:
total 2.6M
-rw-r--r-- 1 root root 2.6M Nov 16 02:54 tokenizer.json
-rw-r--r-- 1 root root  288 Nov 16 02:54 tokenizer_meta.json

data/processed/:
total 123M
-rw-r--r-- 1 root root  247 Nov 16 02:54 dataset_meta.json
-rw-r--r-- 1 root root 120M Nov 16 02:54 train.bin
-rw-r--r-- 1 root root 2.4M Nov 16 02:54 val.bin


In [16]:
# 6) 전체 단위 테스트 (PyTorch Forward/Backward 및 Memmap 테스트 포함)
!PYTHONPATH=src python3 -m unittest discover -s tests -v

test_model_forward_backward_and_generate (test_pipeline.TestSillokPyTorchModel.test_model_forward_backward_and_generate) ... ok
test_dataset_provenance_constants (test_pipeline.TestSillokStandardPipeline.test_dataset_provenance_constants) ... ok
test_parse_surrogate_error_files_cleanly (test_pipeline.TestSillokStandardPipeline.test_parse_surrogate_error_files_cleanly) ... skipped 'Hyojong XML file not present'
test_parse_taejo_xml_and_date_inheritance (test_pipeline.TestSillokStandardPipeline.test_parse_taejo_xml_and_date_inheritance) ... skipped 'Sillok XML data not present'
test_postscript_extraction_and_formatting (test_pipeline.TestSillokStandardPipeline.test_postscript_extraction_and_formatting) ... skipped 'Jeongjong XML file not present'
test_remote_runtime_detection (test_pipeline.TestSillokStandardPipeline.test_remote_runtime_detection) ... ok
test_sanitize_xml_surrogates_decodes_cjk_extension_pair (test_pipeline.TestSillokStandardPipeline.test_sanitize_xml_surrogates_decodes_

In [17]:
# 7) A100 From Scratch 사전학습 실행 (체크포인트는 Google Drive에 자동 심볼릭 링크 저장 및 재개)
!PYTHONPATH=src python3 scripts/04_pretrain.py \
    --config configs/sillok_350m_a100.yaml \
    --drive-ckpt-dir /content/drive/MyDrive/sillok_lm/checkpoints/sillok_350m

Runtime Mode    : Google Colab Remote
Python          : 3.13.16
CPU             : Intel(R) Xeon(R) CPU @ 2.00GHz (2 cores)
System Memory   : 10.86 / 12.67 GiB available
PyTorch         : installed
CUDA / GPU      : Tesla T4 (14.56 GB, bf16=True)
Host Services   : sshd=True, vscode-server=False, cloudflared=inactive
Google Drive    : /content/drive/MyDrive
[remote] Linked checkpoints/sillok_350m -> /content/drive/MyDrive/sillok_lm/checkpoints/sillok_350m
[model] Analytical parameters: 347,132,928
[train] device=cuda dtype=torch.bfloat16 params=347,132,928 tokens/step=262,144 max_steps=20000
W1116 02:55:36.639000 16829 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat

In [18]:
# 8) 학습된 체크포인트로 실록 조건부 생성 테스트
!PYTHONPATH=src python3 scripts/05_generate.py \
    --checkpoint checkpoints/sillok_350m/latest.pt \
    --tokenizer artifacts/tokenizer/tokenizer.json \
    --king "태조실록(太祖實錄)" \
    --date "태조 1년 7월 17일 | 1392-07-17L0 | 丙申" \
    --title-ko "태조가 수창궁에서 왕위에 오르다" \
    --mode hanja

Traceback (most recent call last):
  File "/content/sillok/scripts/05_generate.py", line 49, in <module>
    main()
    ~~~~^^
  File "/content/sillok/scripts/05_generate.py", line 28, in main
    result = generate_from_checkpoint(
        args.checkpoint,
    ...<10 lines>...
        top_p=args.top_p,
    )
  File "/content/sillok/src/sillok/training/generate.py", line 33, in generate_from_checkpoint
    payload = torch.load(checkpoint_path, map_location=target_device, weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1565, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ~~~~^^^^^^^^